___
<a href='https://honglab.ai'><p style="text-align:center;"><img src='https://lh3.googleusercontent.com/lY3ySXooSmwsq5r-mRi7uiypbo0Vez6pmNoQxMFhl9fmZJkRHu5lO2vo7se_0YOzgmDyJif9fi4_z0o3ZFdwd8NVSWG6Ea80uWaf3pOHpR4GHGDV7kaFeuHR3yAjIJjDgfXMxsvw=w2400'  class="center" width="50%" height="50%"/></p></a>
___
<center><em>Content Copyright by HongLab, Inc.</em></center>

# 글자 빈도와 샘플링

**해당 강의:** 파트1 챕터2 「04-02. 글자 빈도 통계」

국어사전 표제어 42만 개에 글자마다 몇 번씩 나오는지 세고, 그 빈도를 확률로 바꿔서 글자를 뽑아 봅니다.
강의에서는 C로 만들었고, 여기서는 같은 일을 파이썬 표준 라이브러리만으로 합니다.

- 실행 시간: 몇 초 (CPU)
- 설치할 것: 없음 (파이썬 표준 라이브러리만 씁니다)

## 데이터 읽기

`data/words.txt`는 표준국어대사전 표제어를 한글만 남기고 정리한 목록입니다. 한 줄에 한 단어씩 들어 있습니다.

In [1]:
words = [line.strip() for line in open("data/words.txt", encoding="utf-8") if line.strip()]

print(f"단어 수: {len(words):,}")
print(words[:10])

단어 수: 420,309
['가', '가가', '가가대소', '가가대소하다', '가가례', '가가린', '가가문전', '가가방', '가가붓자식', '가가성자']


## 글자 빈도 세기

파이썬의 `Counter`는 개수를 세는 딕셔너리입니다. 한글 완성형(가–힣) 글자만 셉니다.

In [2]:
from collections import Counter

freq = Counter(ch for w in words for ch in w if "가" <= ch <= "힣")
total = sum(freq.values())

print(f"한글 완성형 11,172자 중 사전에 쓰인 글자: {len(freq):,}자")
print(f"전체 글자 수: {total:,}\n")
for rank, (ch, cnt) in enumerate(freq.most_common(20), 1):
    print(f"{rank:2d}. {ch}  {cnt:,}")

한글 완성형 11,172자 중 사전에 쓰인 글자: 2,450자
전체 글자 수: 1,551,565

 1. 다  76,241
 2. 하  48,158
 3. 이  25,670
 4. 기  25,462
 5. 리  23,920
 6. 지  19,062
 7. 사  17,600
 8. 자  15,418
 9. 대  14,854
10. 수  13,655
11. 가  12,270
12. 전  11,188
13. 도  10,907
14. 스  10,570
15. 시  10,089
16. 무  9,969
17. 장  9,874
18. 화  9,788
19. 정  9,761
20. 비  9,638


「다」가 압도적인 1위입니다. 사전에는 동사와 형용사가 기본형(「먹다」, 「예쁘다」)으로 실려 있기 때문입니다.
한국어 문장에서 「다」가 가장 많이 쓰인다는 뜻이 아니라 사전이라는 데이터의 특성입니다.

In [3]:
n_da = sum(w.endswith("다") for w in words)
print(f"「다」로 끝나는 표제어: {n_da:,}개 (전체의 {n_da / len(words):.0%})")

「다」로 끝나는 표제어: 71,419개 (전체의 17%)


## 빈도를 확률로

각 글자의 빈도를 전체 글자 수로 나누면 확률이 됩니다. 확률을 모두 더하면 1입니다.

In [4]:
prob = {ch: cnt / total for ch, cnt in freq.items()}

for ch, _ in freq.most_common(5):
    print(f"P({ch}) = {prob[ch]:.4f}")
print("\n확률의 합:", round(sum(prob.values()), 6))

P(다) = 0.0491
P(하) = 0.0310
P(이) = 0.0165
P(기) = 0.0164
P(리) = 0.0154

확률의 합: 1.0


## 확률적 샘플링

확률이 높은 글자는 자주, 낮은 글자는 드물게 뽑습니다. `random.choices`에 글자 목록과 가중치(빈도)를 넘기면 됩니다.

시드를 `42`로 고정해 두었기 때문에 다시 실행해도 같은 글자가 나옵니다. 시드를 바꾸면 다른 글자가 나옵니다.

In [5]:
import random

rng = random.Random(42)
chars = list(freq)
weights = list(freq.values())

for _ in range(10):
    print("".join(rng.choices(chars, weights=weights, k=10)))

악하비은벌갈숟다칙하
관장하거그무관리위가
후난경간황론다다량심
후갯상냥석생알증만리
타하기절다기다비회동
충제강평름창물염사선
꿍굴스꼬신잔기하이강
제릿롱이용수삐투형지
스판리립수관쓱장다하
전행질칙다선잃살쳐만


글자는 사전에서의 비율대로 뽑히지만 뜻은 없습니다. 앞 글자와 상관없이 한 글자씩 따로 뽑기 때문입니다.
앞 글자를 보고 다음 글자를 고르는 바이그램은 `02` 노트북에서 다룹니다.

## 누적 빈도와 이진 탐색

`random.choices` 안에서는 누적 빈도와 이진 탐색으로 글자를 찾습니다. 강의의 C 코드도 같은 방법을 씁니다.

빈도가 「다」 50, 「하」 30, 「는」 20이면 누적 빈도는 50, 80, 100입니다.
0 이상 100 미만의 난수를 하나 뽑아서 0–49면 「다」, 50–79면 「하」, 80–99면 「는」을 고릅니다.

In [6]:
from bisect import bisect_right
from itertools import accumulate

toy_chars = ["다", "하", "는"]
toy_counts = [50, 30, 20]
toy_cum = list(accumulate(toy_counts))
print("누적 빈도:", toy_cum)

for r in [10, 63, 95]:
    print(f"난수 {r:2d} → {toy_chars[bisect_right(toy_cum, r)]}")

누적 빈도: [50, 80, 100]
난수 10 → 다
난수 63 → 하
난수 95 → 는


누적 빈도는 항상 커지는 순서로 쌓이므로 이진 탐색으로 빠르게 찾을 수 있습니다.
같은 방법을 사전의 글자 전체에 적용하고, 10만 번 뽑은 비율이 확률과 맞는지 확인합니다.

In [7]:
import math

cum = list(accumulate(weights))
rng = random.Random(42)

def sample_char():
    r = rng.randrange(cum[-1])           # 0 이상 전체 글자 수 미만의 난수
    return chars[bisect_right(cum, r)]

print("".join(sample_char() for _ in range(30)))
print(f"\n{len(chars):,}자 중 한 글자를 찾는 데 필요한 비교: 최대 {math.ceil(math.log2(len(chars)))}번\n")

sampled = Counter(sample_char() for _ in range(100_000))
for ch, _ in freq.most_common(5):
    print(f"{ch}: 확률 {prob[ch]:.2%}   10만 번 중 뽑힌 비율 {sampled[ch] / 100_000:.2%}")

복고하당정이부놔성삐벌방두업하하자과이디접하바강륨슬으벌레내

2,450자 중 한 글자를 찾는 데 필요한 비교: 최대 12번

다: 확률 4.91%   10만 번 중 뽑힌 비율 4.90%
하: 확률 3.10%   10만 번 중 뽑힌 비율 3.08%
이: 확률 1.65%   10만 번 중 뽑힌 비율 1.62%
기: 확률 1.64%   10만 번 중 뽑힌 비율 1.60%
리: 확률 1.54%   10만 번 중 뽑힌 비율 1.57%


## 탐욕적 디코딩

매번 확률이 가장 높은 글자만 고르면 어떻게 될까요? 이 방식을 탐욕적 디코딩(greedy decoding)이라고 합니다.

In [8]:
top = max(freq, key=freq.get)
print("".join(top for _ in range(20)))

다다다다다다다다다다다다다다다다다다다다


같은 글자만 반복됩니다. 현대 LLM도 탐욕적 디코딩을 쓰면 같은 말을 되풀이하기 쉬워서 확률적 샘플링을 씁니다.

## [보충] 한글은 3바이트

UTF-8에서 한글 한 글자는 3바이트입니다. 각 바이트의 앞쪽 몇 비트는 몇 바이트짜리 글자인지 알려 주는 표시이고, 나머지 비트를 이어 붙이면 코드포인트(유니코드 번호)가 됩니다.
파이썬의 `ord`가 해 주는 일을 C 코드에서는 비트 연산으로 직접 했습니다.

In [9]:
ch = "가"
b = ch.encode("utf-8")
print("바이트:", [hex(x) for x in b])
print("2진수:", [format(x, "08b") for x in b])

codepoint = ((b[0] & 0x0F) << 12) | ((b[1] & 0x3F) << 6) | (b[2] & 0x3F)
print(f"\n비트 연산으로 구한 코드포인트: {codepoint} ({hex(codepoint)})")
print(f"ord('{ch}'): {ord(ch)}")

바이트: ['0xea', '0xb0', '0x80']
2진수: ['11101010', '10110000', '10000000']

비트 연산으로 구한 코드포인트: 44032 (0xac00)
ord('가'): 44032


## 정리

- 빈도를 전체 합으로 나누면 확률이 됩니다.
- 확률적 샘플링은 확률대로 다양하게 뽑고, 탐욕적 디코딩은 확률이 가장 높은 글자만 골라 같은 글자를 반복합니다.
- 한 글자씩 따로 뽑으면 뜻 없는 글자의 나열이 됩니다.

`02_엔그램으로단어만들기`에서는 앞 글자를 보고 다음 글자를 고릅니다.